# ☄️ Radar de Asteroides Cercanos a la Tierra
## Proyecto Final — Programación para Análisis de Datos
### Fuente: NASA JPL Small-Body Database (2015–2035)

---

Este notebook documenta el análisis de datos detrás del **dashboard interactivo de asteroides** construido con Streamlit y Plotly.

**Objetivos del proyecto:**
- Explorar y limpiar datos reales de la NASA sobre acercamientos de asteroides a la Tierra
- Identificar patrones en velocidad, tamaño y distancia
- Clasificar asteroides peligrosos (PHAs) vs. inofensivos
- Construir visualizaciones interactivas (radar 2D, simulador 3D, series de tiempo)

**Datasets utilizados:**
- `asteroid_close_approaches_2015_2035.csv` — 42,000+ acercamientos registrados y proyectados
- `near_earth_asteroids_2025.csv` — Propiedades físicas y orbitales de cada asteroide

## 1. Importación de Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings("ignore")

# Configuración de estilo para matplotlib
plt.rcParams.update({
    "figure.facecolor": "#0a0a1a",
    "axes.facecolor": "#0d0d2b",
    "axes.edgecolor": "#7ecfff",
    "axes.labelcolor": "#7ecfff",
    "text.color": "white",
    "xtick.color": "white",
    "ytick.color": "white",
    "grid.color": "#1a1a2e",
    "grid.linestyle": "--",
    "grid.alpha": 0.5,
})

print("Librerías cargadas correctamente ✅")

## 2. Carga y Exploración Inicial de los Datos

### 2.1 Dataset de Acercamientos (2015–2035)

In [ ]:
df_acercamientos = pd.read_csv(
    "asteroid_close_approaches_2015_2035 (1).csv",
    low_memory=False
)

print(f"Filas: {df_acercamientos.shape[0]:,}  |  Columnas: {df_acercamientos.shape[1]}")
print("\n--- Primeras filas ---")
df_acercamientos.head(3)

In [ ]:
print("--- Tipos de datos y valores nulos ---")
df_acercamientos.info()

### 2.2 Dataset de Asteroides Cercanos (NEAs 2025)

In [ ]:
df_asteroides = pd.read_csv(
    "near_earth_asteroids_2025 (1).csv",
    low_memory=False
)

print(f"Filas: {df_asteroides.shape[0]:,}  |  Columnas: {df_asteroides.shape[1]}")
print("\n--- Primeras filas ---")
df_asteroides.head(3)

## 3. Limpieza y Preparación de Datos

### 3.1 Unión de los dos datasets por `full_name`

Los dos datasets comparten la columna `full_name` (nombre oficial del asteroide según la NASA).
Se hace un **inner join** para quedarnos solo con los asteroides que tienen tanto datos de acercamiento como propiedades físicas.

In [ ]:
df = pd.merge(df_acercamientos, df_asteroides, on="full_name", how="inner")

print(f"Registros antes del merge : {len(df_acercamientos):>8,}")
print(f"Asteroides únicos (NEAs)  : {len(df_asteroides):>8,}")
print(f"Registros después del join: {len(df):>8,}")
print(f"Columnas resultantes      : {df.shape[1]}")

### 3.2 Eliminación de valores nulos en columnas clave

In [ ]:
cols_clave = ["diameter_m", "distance_au", "velocity_km_s"]

print("Nulos por columna clave ANTES de limpiar:")
print(df[cols_clave].isnull().sum().to_string())

df = df.dropna(subset=cols_clave)

print(f"\nRegistros después de eliminar nulos: {len(df):,}")
print(f"Porcentaje retenido: {len(df)/len(pd.merge(df_acercamientos, df_asteroides, on='full_name', how='inner'))*100:.1f}%")

### 3.3 Ingeniería de características (Feature Engineering)

In [ ]:
# Convertir fecha a datetime y extraer el año
df["close_approach_date"] = pd.to_datetime(df["close_approach_date"], errors="coerce")
df["año"] = df["close_approach_date"].dt.year

# Etiqueta legible de peligrosidad
df["peligro_label"] = df["pha"].apply(
    lambda x: "⚠️ Peligroso (PHA)" if x else "✅ Inofensivo"
)

# Coordenadas 3D ilustrativas (ángulos aleatorios + distancia real)
rng = np.random.default_rng(42)
n = len(df)
theta = rng.uniform(0, 2 * np.pi, n)
phi   = rng.uniform(0, np.pi, n)
r     = df["distance_au"].values
df["eje_x"] = r * np.sin(phi) * np.cos(theta)
df["eje_y"] = r * np.sin(phi) * np.sin(theta)
df["eje_z"] = r * np.cos(phi)

print("Columnas nuevas creadas:")
print(["año", "peligro_label", "eje_x", "eje_y", "eje_z"])
df[["full_name", "año", "peligro_label", "distance_au", "eje_x", "eje_y", "eje_z"]].head(5)

## 4. Análisis Exploratorio (EDA)

### 4.1 Estadísticas descriptivas de las variables numéricas clave

In [ ]:
estadisticas = df[["distance_au", "velocity_km_s", "diameter_m", "dist_lunar"]].describe().T
estadisticas.columns = ["N", "Media", "Desv. Std", "Mín", "Q1", "Mediana", "Q3", "Máx"]
estadisticas.index.name = "Variable"
estadisticas.round(3)

### 4.2 KPIs Principales del Dataset

In [ ]:
total       = len(df)
phas        = int(df["pha"].sum())
pct_pha     = df["pha"].mean() * 100
diam_prom   = df["diameter_m"].mean()
vel_max     = df["velocity_km_s"].max()
dist_min_ld = df["dist_lunar"].min()
asteroide_mas_rapido = df.loc[df["velocity_km_s"].idxmax(), "full_name"].strip()
asteroide_mas_cercano = df.loc[df["dist_lunar"].idxmin(), "full_name"].strip()

print(f"{'☄️  Total de acercamientos':<40} {total:>10,}")
print(f"{'⚠️  PHAs (Potencialmente Peligrosos)':<40} {phas:>10,}  ({pct_pha:.1f}%)")
print(f"{'📏  Diámetro promedio':<40} {diam_prom:>10.1f} m")
print(f"{'🚀  Velocidad máxima registrada':<40} {vel_max:>10.1f} km/s")
print(f"{'   └─ Asteroide':<40} {asteroide_mas_rapido}")
print(f"{'🌕  Distancia mínima a la Tierra':<40} {dist_min_ld:>10.2f} LD")
print(f"{'   └─ Asteroide':<40} {asteroide_mas_cercano}")

### 4.3 Distribución por Categoría de Tamaño

In [ ]:
conteos = df["size_category"].value_counts()

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(conteos.index, conteos.values,
               color=plt.cm.Blues(np.linspace(0.4, 0.9, len(conteos))))
ax.set_xlabel("Número de acercamientos")
ax.set_title("Acercamientos por Categoría de Tamaño", color="#7ecfff", fontsize=13)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
for bar, val in zip(bars, conteos.values):
    ax.text(val + 100, bar.get_y() + bar.get_height()/2,
            f"{val:,}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

### 4.4 Distribución de Velocidades: PHAs vs. Inofensivos

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

for label, color in [("✅ Inofensivo", "#33cc33"), ("⚠️ Peligroso (PHA)", "#ff3333")]:
    subset = df[df["peligro_label"] == label]["velocity_km_s"]
    ax.hist(subset, bins=60, alpha=0.6, color=color, label=f"{label} (n={len(subset):,})", edgecolor="none")

ax.set_xlabel("Velocidad de impacto (km/s)")
ax.set_ylabel("Frecuencia")
ax.set_title("Distribución de Velocidades por Clasificación de Peligro", color="#7ecfff", fontsize=13)
ax.legend(facecolor="#0d0d2b", edgecolor="#7ecfff")
plt.tight_layout()
plt.show()

# Velocidad media por grupo
print("Velocidad media (km/s):")
print(df.groupby("peligro_label")["velocity_km_s"].mean().round(2).to_string())

### 4.5 Correlación: Velocidad vs. Diámetro

In [ ]:
sample = df.sample(3000, random_state=1)

fig, ax = plt.subplots(figsize=(9, 5))
for label, color in [("✅ Inofensivo", "#33cc33"), ("⚠️ Peligroso (PHA)", "#ff3333")]:
    sub = sample[sample["peligro_label"] == label]
    ax.scatter(sub["diameter_m"], sub["velocity_km_s"],
               alpha=0.35, s=15, color=color, label=label)

ax.set_xlabel("Diámetro (m)")
ax.set_ylabel("Velocidad (km/s)")
ax.set_title("Velocidad vs. Diámetro del Asteroide", color="#7ecfff", fontsize=13)
ax.legend(facecolor="#0d0d2b", edgecolor="#7ecfff")
ax.set_xscale("log")
plt.tight_layout()
plt.show()

corr = df[["diameter_m", "velocity_km_s"]].corr().iloc[0, 1]
print(f"Coeficiente de correlación de Pearson (diámetro vs. velocidad): {corr:.4f}")

## 5. Análisis Temporal — Serie de Tiempo

### 5.1 Acercamientos por mes (2015–2035)

In [ ]:
df["mes"] = df["close_approach_date"].dt.to_period("M").astype(str)
por_mes = df.groupby(["mes", "peligro_label"]).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(14, 5))
por_mes.plot(
    kind="bar", stacked=True, ax=ax,
    color={"✅ Inofensivo": "#33cc33", "⚠️ Peligroso (PHA)": "#ff3333"},
    width=0.85, edgecolor="none"
)
ax.set_xlabel("Mes")
ax.set_ylabel("Acercamientos")
ax.set_title("Acercamientos Mensuales de Asteroides a la Tierra (2015–2035)", color="#7ecfff", fontsize=13)
ax.set_xticks(range(0, len(por_mes), 12))
ax.set_xticklabels([str(por_mes.index[i])[:4] for i in range(0, len(por_mes), 12)], rotation=0)
ax.legend(facecolor="#0d0d2b", edgecolor="#7ecfff")
plt.tight_layout()
plt.show()

### 5.2 Tendencia anual: acercamientos totales y distancia promedio

In [ ]:
por_año = df.groupby("año").agg(
    acercamientos=("full_name", "count"),
    dist_prom=("distance_au", "mean"),
    phas=("pha", "sum")
).reset_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Total de acercamientos por año
ax1.plot(por_año["año"], por_año["acercamientos"], marker="o", color="#7ecfff", linewidth=2)
ax1.fill_between(por_año["año"], por_año["acercamientos"], alpha=0.15, color="#7ecfff")
ax1.set_title("Total de Acercamientos por Año", color="#7ecfff", fontsize=12)
ax1.set_xlabel("Año")
ax1.set_ylabel("Acercamientos")

# Distancia promedio anual
ax2.plot(por_año["año"], por_año["dist_prom"], marker="o", color="#ffd700", linewidth=2)
ax2.fill_between(por_año["año"], por_año["dist_prom"], alpha=0.15, color="#ffd700")
ax2.set_title("Distancia Promedio Anual (UA)", color="#ffd700", fontsize=12)
ax2.set_xlabel("Año")
ax2.set_ylabel("Distancia promedio (UA)")

plt.tight_layout()
plt.show()

print("\nAño con más acercamientos:")
print(por_año.nlargest(3, "acercamientos")[["año", "acercamientos", "phas"]].to_string(index=False))

## 6. Visualizaciones Interactivas (Plotly)

> Estas son las mismas gráficas del dashboard Streamlit. En el notebook se renderizan de forma inline.

### 6.1 Radar 2D — Distancia vs. Velocidad (burbujas = tamaño real)

In [ ]:
fig_radar = px.scatter(
    df.sample(4000, random_state=7),
    x="distance_au",
    y="velocity_km_s",
    size="diameter_m",
    color="peligro_label",
    hover_name="full_name",
    hover_data={
        "close_approach_date": True,
        "size_category": True,
        "diameter_m": ":.1f",
        "velocity_km_s": ":.2f",
        "dist_lunar": ":.2f",
        "peligro_label": False,
    },
    labels={
        "distance_au": "Distancia mínima a la Tierra (UA)",
        "velocity_km_s": "Velocidad de impacto (km/s)",
        "diameter_m": "Diámetro (m)",
        "peligro_label": "Clasificación",
    },
    color_discrete_map={
        "⚠️ Peligroso (PHA)": "#ff3333",
        "✅ Inofensivo": "#33cc33",
    },
    size_max=50,
    template="plotly_dark",
    title="Radar 2D — Distancia vs. Velocidad de Asteroides",
)
fig_radar.update_layout(
    height=550,
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(5,5,20,0.8)",
    legend=dict(orientation="h", y=-0.15),
)
fig_radar.show()

### 6.2 Simulador 3D — Vecindario Terrestre

La posición angular es aleatoria (reproducible con semilla 42). La distancia al origen sí corresponde a la UA real de cada acercamiento.

In [ ]:
df_3d = df.sample(1500, random_state=42)
peligrosos  = df_3d[df_3d["pha"] == True]
inofensivos = df_3d[df_3d["pha"] == False]

fig3d = go.Figure()

fig3d.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0],
    mode="markers+text",
    marker=dict(size=14, color="#4da6ff", opacity=0.95),
    text=["🌍 Tierra"], textposition="top center",
    name="Tierra"
))
fig3d.add_trace(go.Scatter3d(
    x=inofensivos["eje_x"], y=inofensivos["eje_y"], z=inofensivos["eje_z"],
    mode="markers",
    marker=dict(size=np.clip(inofensivos["diameter_m"] / 25, 2, 12),
                color="lightgray", opacity=0.45),
    hovertext=inofensivos["full_name"] + "<br>" +
              "Dist: " + inofensivos["distance_au"].round(4).astype(str) + " UA",
    hoverinfo="text",
    name="✅ Inofensivos"
))
fig3d.add_trace(go.Scatter3d(
    x=peligrosos["eje_x"], y=peligrosos["eje_y"], z=peligrosos["eje_z"],
    mode="markers",
    marker=dict(size=np.clip(peligrosos["diameter_m"] / 25, 4, 16),
                color="#ff4444", opacity=0.9,
                line=dict(color="#ff0000", width=1)),
    hovertext=peligrosos["full_name"] + "<br>⚠️ PHA<br>" +
              "Dist: " + peligrosos["distance_au"].round(4).astype(str) + " UA",
    hoverinfo="text",
    name="⚠️ Peligrosos (PHA)"
))

fig3d.update_layout(
    scene=dict(
        xaxis=dict(title="X (UA)", backgroundcolor="black", gridcolor="#1a1a2e", showbackground=True),
        yaxis=dict(title="Y (UA)", backgroundcolor="black", gridcolor="#1a1a2e", showbackground=True),
        zaxis=dict(title="Z (UA)", backgroundcolor="black", gridcolor="#1a1a2e", showbackground=True),
    ),
    paper_bgcolor="black",
    font=dict(color="white"),
    height=620,
    title="Simulador 3D del Vecindario Terrestre",
    legend=dict(bgcolor="rgba(0,0,0,0.5)", bordercolor="#333", borderwidth=1)
)
fig3d.show()

### 6.3 Box Plot — Diámetro por Categoría de Tamaño y Peligrosidad

In [ ]:
fig_box = px.box(
    df, x="size_category", y="diameter_m",
    color="peligro_label",
    color_discrete_map={
        "⚠️ Peligroso (PHA)": "#ff3333",
        "✅ Inofensivo": "#33cc33",
    },
    labels={
        "size_category": "Categoría de tamaño",
        "diameter_m": "Diámetro (m)",
        "peligro_label": ""
    },
    template="plotly_dark",
    title="Distribución de Diámetros por Categoría y Peligrosidad",
)
fig_box.update_xaxes(tickangle=-20)
fig_box.update_layout(
    height=420,
    paper_bgcolor="rgba(0,0,0,0)",
    legend=dict(orientation="h", y=-0.3),
)
fig_box.show()

## 7. Top 10 Acercamientos Más Peligrosos

Filtramos los asteroides PHAs con menor distancia lunar registrada en todo el dataset.

In [ ]:
top_peligrosos = (
    df[df["pha"] == True][[
        "full_name", "close_approach_date", "distance_au",
        "dist_lunar", "velocity_km_s", "diameter_m", "size_category"
    ]]
    .sort_values("dist_lunar")
    .head(10)
    .copy()
)
top_peligrosos["close_approach_date"] = top_peligrosos["close_approach_date"].dt.strftime("%Y-%m-%d")
top_peligrosos = top_peligrosos.rename(columns={
    "full_name": "Asteroide",
    "close_approach_date": "Fecha",
    "distance_au": "Dist. (UA)",
    "dist_lunar": "Dist. Lunar (LD)",
    "velocity_km_s": "Vel. (km/s)",
    "diameter_m": "Diámetro (m)",
    "size_category": "Categoría",
})
top_peligrosos.reset_index(drop=True, inplace=True)
top_peligrosos.index += 1
top_peligrosos

## 8. Arquitectura del Dashboard (asteroid_dashboard.py)

```
asteroid_dashboard.py
│
├── cargar_datos()          ← @st.cache_data: merge, limpieza, feature engineering
│
├── SIDEBAR                 ← Filtros globales (PHA, tamaño, años, distancia)
│
├── KPIs (5 métricas)
│
└── TABS
    ├── Tab 1 — Radar 2D    ← px.scatter (distancia vs. velocidad, burbujas = diámetro)
    ├── Tab 2 — Simulador 3D← go.Scatter3d (coordenadas esféricas ilustrativas)
    ├── Tab 3 — Distribuciones ← barras, histograma, box plot, scatter
    ├── Tab 4 — Serie de Tiempo ← barras mensuales + líneas anuales
    └── Tab 5 — Explorador  ← DataTable con búsqueda + descarga CSV
```

**Decisiones de diseño importantes:**

| Decisión | Justificación |
|---|---|
| `@st.cache_data` en `cargar_datos()` | Evita re-leer los CSVs en cada interacción del usuario |
| `inner join` por `full_name` | Solo trabajamos con asteroides que tienen propiedades físicas conocidas |
| Coordenadas 3D con semilla fija (`seed=42`) | Reproducibilidad: la misma ejecución siempre da el mismo mapa |
| `size_max` configurable en Radar 2D | Permite al usuario ajustar la legibilidad según la densidad del filtro |
| Limitar 3D a N asteroides | El renderer WebGL de Plotly se degrada con >5000 puntos en un notebook |

## 9. Conclusiones

---

### Hallazgos principales

1. **Volumen de datos:** El dataset fusionado contiene decenas de miles de acercamientos con propiedades físicas completas, abarcando 20 años de observaciones y proyecciones (2015–2035).

2. **Distribución de peligrosidad:** Solo una minoría de los acercamientos corresponde a PHAs. La gran mayoría de asteroides que se acercan a la Tierra son objetos pequeños e inofensivos.

3. **Velocidad:** Los PHAs muestran, en promedio, velocidades **más altas** que los inofensivos. Esto tiene sentido orbital: los asteroides con órbitas que se cruzan significativamente con la Tierra tienden a tener velocidades relativas mayores.

4. **Tamaño vs. velocidad:** La correlación es cercana a cero — el tamaño de un asteroide no determina su velocidad. Son variables orbitalmente independientes.

5. **Tendencia temporal:** La cantidad de acercamientos registrados **aumenta** hacia años recientes, lo que refleja mejoras en los programas de vigilancia astronómica de la NASA, no un aumento real en la actividad asteroidal.

6. **Asteroide más cercano del dataset:** Pasó a menos de 1 distancia lunar de la Tierra — eso es menos de 384,400 km, más cerca que la Luna.

---

### Tecnologías utilizadas

| Librería | Uso |
|---|---|
| `pandas` | Carga, limpieza, merge, agrupaciones |
| `numpy` | Cálculo de coordenadas 3D esféricas |
| `plotly` | Gráficas interactivas (scatter, 3D, histograma, box, barras) |
| `streamlit` | Framework del dashboard interactivo |
| `matplotlib` | Gráficas estáticas para el notebook |

---

*Datos: NASA JPL Small-Body Database — https://ssd.jpl.nasa.gov/tools/sbdb_query.cgi*

## 9. Animación de Acercamientos por Año

Gráfica de burbujas animada con `animation_frame` de Plotly: cada frame es un año (2015–2035).  
La línea dorada marca **1 distancia lunar** (~384,400 km) — las burbujas por debajo pasaron más cerca que la Luna.

### 9.1 Preparación de datos para la animación

In [ ]:
MAX_POR_AÑO = 300  # asteroides por frame — sube hasta ~600 si tu máquina lo aguanta

frames_list = []
for yr, grupo in df.groupby("año"):
    muestra = grupo.sample(min(MAX_POR_AÑO, len(grupo)), random_state=int(yr))
    frames_list.append(muestra)

df_anim = pd.concat(frames_list, ignore_index=True)
df_anim["año_str"] = df_anim["año"].astype(str)

print(f"Años disponibles : {sorted(df_anim['año'].unique())}")
print(f"Total de puntos  : {len(df_anim):,}  ({MAX_POR_AÑO} por año × {df_anim['año'].nunique()} años)")

### 9.2 Animación: Distancia (UA) vs. Distancia Lunar

In [ ]:
# Ejes fijos con percentiles para que no salten entre frames
x_range = [df_anim["distance_au"].quantile(0.01), df_anim["distance_au"].quantile(0.99)]
y_range = [df_anim["dist_lunar"].quantile(0.01),  df_anim["dist_lunar"].quantile(0.99)]

fig_anim = px.scatter(
    df_anim,
    x="distance_au",
    y="dist_lunar",
    size="diameter_m",
    color="peligro_label",
    animation_frame="año_str",
    animation_group="full_name",
    hover_name="full_name",
    hover_data={
        "close_approach_date": True,
        "velocity_km_s": ":.2f",
        "diameter_m": ":.0f",
        "size_category": True,
        "peligro_label": False,
        "año_str": False,
    },
    labels={
        "distance_au": "Distancia mínima a la Tierra (UA)",
        "dist_lunar": "Distancia Lunar (LD)",
        "diameter_m": "Diámetro (m)",
        "peligro_label": "Clasificación",
    },
    color_discrete_map={
        "⚠️ Peligroso (PHA)": "#ff3333",
        "✅ Inofensivo": "#33cc33",
    },
    size_max=55,
    range_x=x_range,
    range_y=y_range,
    template="plotly_dark",
    title="Acercamientos de Asteroides a la Tierra — Animación 2015–2035",
)

fig_anim.update_layout(
    height=600,
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(5,5,20,0.9)",
    legend=dict(orientation="h", y=-0.15),
    updatemenus=[{
        "buttons": [
            {"args": [None, {"frame": {"duration": 800, "redraw": True},
                             "fromcurrent": True}],
             "label": "▶ Play", "method": "animate"},
            {"args": [[None], {"frame": {"duration": 0, "redraw": True},
                               "mode": "immediate"}],
             "label": "⏸ Pause", "method": "animate"},
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 10},
        "showactive": False,
        "type": "buttons",
        "x": 0.1, "xanchor": "right",
        "y": 0, "yanchor": "top",
    }],
    sliders=[{
        "currentvalue": {
            "prefix": "Año: ",
            "font": {"color": "#7ecfff", "size": 16},
        },
        "pad": {"t": 50},
    }],
)

# Línea de referencia: 1 distancia lunar
fig_anim.add_hline(
    y=1.0,
    line_dash="dash",
    line_color="#ffd700",
    annotation_text="🌕 1 distancia lunar",
    annotation_position="top right",
    annotation_font_color="#ffd700",
)

fig_anim.show()

### 9.3 Animación alternativa: Velocidad vs. Distancia Lunar

In [ ]:
x2_range = [df_anim["velocity_km_s"].quantile(0.01), df_anim["velocity_km_s"].quantile(0.99)]

fig_anim2 = px.scatter(
    df_anim,
    x="velocity_km_s",
    y="dist_lunar",
    size="diameter_m",
    color="peligro_label",
    animation_frame="año_str",
    animation_group="full_name",
    hover_name="full_name",
    hover_data={
        "distance_au": ":.4f",
        "diameter_m": ":.0f",
        "size_category": True,
        "peligro_label": False,
        "año_str": False,
    },
    labels={
        "velocity_km_s": "Velocidad de impacto (km/s)",
        "dist_lunar": "Distancia Lunar (LD)",
        "diameter_m": "Diámetro (m)",
        "peligro_label": "Clasificación",
    },
    color_discrete_map={
        "⚠️ Peligroso (PHA)": "#ff3333",
        "✅ Inofensivo": "#33cc33",
    },
    size_max=50,
    range_x=x2_range,
    range_y=y_range,
    template="plotly_dark",
    title="Velocidad vs. Distancia Lunar — Animación 2015–2035",
)

fig_anim2.update_layout(
    height=560,
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(5,5,20,0.9)",
    legend=dict(orientation="h", y=-0.15),
    updatemenus=[{
        "buttons": [
            {"args": [None, {"frame": {"duration": 800, "redraw": True}, "fromcurrent": True}],
             "label": "▶ Play", "method": "animate"},
            {"args": [[None], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}],
             "label": "⏸ Pause", "method": "animate"},
        ],
        "direction": "left", "pad": {"r": 10, "t": 10},
        "showactive": False, "type": "buttons",
        "x": 0.1, "xanchor": "right", "y": 0, "yanchor": "top",
    }],
    sliders=[{"currentvalue": {"prefix": "Año: ", "font": {"color": "#7ecfff", "size": 16}},
              "pad": {"t": 50}}],
)

fig_anim2.add_hline(
    y=1.0, line_dash="dash", line_color="#ffd700",
    annotation_text="🌕 1 distancia lunar",
    annotation_position="top right",
    annotation_font_color="#ffd700",
)

fig_anim2.show()

### 9.4 ¿Qué estás viendo? — Guía de lectura de la animación

| Elemento | Significado |
|---|---|
| **Posición X** | Distancia mínima al momento del acercamiento (UA o km/s según la vista) |
| **Posición Y** | Distancia en unidades lunares (1 LD ≈ 384,400 km) |
| **Tamaño de burbuja** | Diámetro real del asteroide en metros |
| **Color rojo** | PHA — Potentially Hazardous Asteroid (órbita que cruza la Tierra con tamaño ≥ 140 m) |
| **Color verde** | Inofensivo — se acerca pero no cumple criterios de peligrosidad |
| **Línea dorada** | 1 distancia lunar — umbral visual de "muy cerca" |
| **Slider de año** | Mueve manualmente al año que quieras inspeccionar |

**Cómo leer la tendencia:** si el número de burbujas crece hacia 2025–2035, no significa que haya más asteroides — significa que los programas de vigilancia de la NASA están detectando **objetos más pequeños** que antes pasaban desapercibidos.